# TTT for TTSF

## Install Packages:

In [1]:
!pip install gluonts chronos-forecasting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 5.2 MB/s eta 0:00:00


## Import Packages:

In [26]:
import torch
import numpy as np
from chronos import Chronos2Pipeline
from gluonts.dataset.repository import get_dataset
import json
from pathlib import Path
import wandb
import copy
import math
import torch.nn.functional as F
from tqdm import tqdm
import contextlib
import io
from itertools import product

## Inspecting Chronos 2 Architecture to Find the Embedding Layer for TTT

In [ ]:
print("Loading Chronos-2 pipeline...")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nDevice: {device}")

pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=device,
    torch_dtype=torch.float32,
)

model = pipeline.model
print(f"\nModel type: {type(model)}")
print(f"Model class name: {type(model).__name__}")

# Overall structure (top-level children only)
print("\n" + "=" * 70)
print("TOP-LEVEL MODULES")
print("=" * 70)
for name, child in model.named_children():
    params = sum(p.numel() for p in child.parameters())
    print(f"  {name}: {type(child).__name__} ({params:,} params)")

# All named modules with param counts
print("\n" + "=" * 70)
print("ALL NAMED MODULES (with parameters)")
print("=" * 70)
for name, module in model.named_modules():
    params = sum(p.numel() for p in module.parameters(recurse=False))
    if params > 0:
        print(f"  {name}: {type(module).__name__} ({params:,} params)")

# Candidate embedding layers
keywords = ["embed", "proj", "input", "patch", "token"]
print("\n" + "=" * 70)
print(f"CANDIDATE EMBEDDING LAYERS (keywords: {keywords})")
print("=" * 70)
for name, module in model.named_modules():
    if any(kw in name.lower() for kw in keywords):
        params = sum(p.numel() for p in module.parameters())
        print(f"\n  {name}: {type(module).__name__} ({params:,} params)")
        for pname, param in module.named_parameters(recurse=False):
            print(f"    .{pname}: shape={list(param.shape)}, dtype={param.dtype}")

# All named parameters for full picture
print("\n" + "=" * 70)
print("ALL NAMED PARAMETERS")
print("=" * 70)
total = 0
for name, param in model.named_parameters():
    total += param.numel()
    print(f"  {name}: shape={list(param.shape)} ({param.numel():,} params)")
print(f"\nTotal parameters: {total:,}")

Loading Chronos-2 pipeline...

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]


Model type: <class 'chronos.chronos2.model.Chronos2Model'>
Model class name: Chronos2Model

TOP-LEVEL MODULES
  shared: Embedding (1,536 params)
  input_patch_embedding: ResidualBlock (2,548,224 params)
  patch: Patch (0 params)
  instance_norm: InstanceNorm (0 params)
  encoder: Chronos2Encoder (113,274,624 params)
  output_patch_embedding: ResidualBlock (3,653,280 params)

ALL NAMED MODULES (with parameters)
  shared: Embedding (1,536 params)
  input_patch_embedding.hidden_layer: Linear (150,528 params)
  input_patch_embedding.output_layer: Linear (2,360,064 params)
  input_patch_embedding.residual_layer: Linear (37,632 params)
  encoder.block.0.layer.0.self_attention.q: Linear (589,824 params)
  encoder.block.0.layer.0.self_attention.k: Linear (589,824 params)
  encoder.block.0.layer.0.self_attention.v: Linear (589,824 params)
  encoder.block.0.layer.0.self_attention.o: Linear (589,824 params)
  encoder.block.0.layer.0.layer_norm: Chronos2LayerNorm (768 params)
  encoder.block.0.la

## Download ETTh1 dataset using GluonTS

In [3]:
def get_etth1():
  """Download and return the ETTh1 dataset.

  Returns the GluonTS dataset object for "ett_small_1h" which contains
  hourly Electricity Transformer Temperature data (~17,000 timesteps,
  7 features). The oil temperature (OT) column is used as target.
  """
  dataset = get_dataset("ett_small_1h")
  return dataset

In [4]:
dataset = get_etth1()

print("=== ETTh1 Dataset Stats ===")
print(f"Frequency: {dataset.metadata.freq}")
print(f"Prediction length (default): {dataset.metadata.prediction_length}")

# Test set stats
test_entries = list(dataset.test)
print(f"\nTest set:")
print(f"  Number of time series: {len(test_entries)}")
for i, entry in enumerate(test_entries):
    ts = entry["target"]
    print(f"  Series {i}: length={len(ts)}, start={entry['start']}")

# Train set stats
train_entries = list(dataset.train)
print(f"\nTrain set:")
print(f"  Number of time series: {len(train_entries)}")
for i, entry in enumerate(train_entries):
    ts = entry["target"]
    print(f"  Series {i}: length={len(ts)}, start={entry['start']}")


=== ETTh1 Dataset Stats ===
Frequency: 1h
Prediction length (default): 24

Test set:
  Number of time series: 14
  Series 0: length=17420, start=2016-07-01 00:00
  Series 1: length=17420, start=2016-07-01 00:00
  Series 2: length=17420, start=2016-07-01 00:00
  Series 3: length=17420, start=2016-07-01 00:00
  Series 4: length=17420, start=2016-07-01 00:00
  Series 5: length=17420, start=2016-07-01 00:00
  Series 6: length=17420, start=2016-07-01 00:00
  Series 7: length=17420, start=2016-07-01 00:00
  Series 8: length=17420, start=2016-07-01 00:00
  Series 9: length=17420, start=2016-07-01 00:00
  Series 10: length=17420, start=2016-07-01 00:00
  Series 11: length=17420, start=2016-07-01 00:00
  Series 12: length=17420, start=2016-07-01 00:00
  Series 13: length=17420, start=2016-07-01 00:00

Train set:
  Number of time series: 14
  Series 0: length=17396, start=2016-07-01 00:00
  Series 1: length=17396, start=2016-07-01 00:00
  Series 2: length=17396, start=2016-07-01 00:00
  Series 3

## Convert GluonTS dataset to tensors suitable for Chronos-2

In [5]:
def prepare_data(dataset, context_length=512, prediction_length=96):
  """Create sliding window samples from the dataset test set.

  Takes the test split of a GluonTS dataset and generates
  (context, target) pairs using a sliding window.

  Args:
      dataset: GluonTS dataset object (has .test attribute).
      context_length: Number of past timesteps for context.
      prediction_length: Number of future timesteps to predict.

  Returns:
      contexts: List of torch.Tensor, each shape (context_length,).
      targets: List of torch.Tensor, each shape (prediction_length,).
  """
  contexts = []
  targets = []

  window_size = context_length + prediction_length

  for entry in dataset.test:
      ts = np.array(entry["target"], dtype=np.float32)

      if len(ts) < window_size:
          continue

      # Slide over the time series with non-overlapping steps of prediction_length
      for start in range(0, len(ts) - window_size + 1, prediction_length):
          context = torch.tensor(ts[start : start + context_length])
          target = torch.tensor(ts[start + context_length : start + window_size])
          contexts.append(context)
          targets.append(target)

  return contexts, targets

In [6]:
dataset = get_etth1()
contexts, targets = prepare_data(dataset, context_length=512, prediction_length=96)

print("=== ETTh1 DataLoader Stats ===")
print(f"Number of samples: {len(contexts)}")
if contexts:
    print(f"Context shape: {contexts[0].shape}")
    print(f"Target shape:  {targets[0].shape}")
    print(f"Context dtype: {contexts[0].dtype}")
    print(f"\nFirst context - min: {contexts[0].min():.4f}, max: {contexts[0].max():.4f}")
    print(f"First target  - min: {targets[0].min():.4f}, max: {targets[0].max():.4f}")


=== ETTh1 DataLoader Stats ===
Number of samples: 2464
Context shape: torch.Size([512])
Target shape:  torch.Size([96])
Context dtype: torch.float32

First context - min: 4.2200, max: 19.1560
First target  - min: 3.2150, max: 18.6200


## Evaluation metrics for time series forecasting

In [7]:
def _to_point_forecast(pred):
  """Convert predictions to point forecast.

  If pred has 3 dimensions (num_samples, batch, pred_len), take the median
  across samples to get a single point forecast. This handles Chronos-2's
  probabilistic output which returns multiple forecast samples.
  """
  if isinstance(pred, np.ndarray):
      pred = torch.tensor(pred)

  if pred.ndim == 3:
      # (num_samples, batch, pred_len) -> median over samples
      pred = pred.median(dim=0).values

  return pred.float()


In [8]:
def compute_mse(pred, target):
  """Mean Squared Error between prediction and target.

  Args:
      pred: Predictions, shape (batch, pred_len) or (num_samples, batch, pred_len).
      target: Ground truth, shape (batch, pred_len).

  Returns:
      Scalar MSE value.
  """
  pred = _to_point_forecast(pred)
  if isinstance(target, np.ndarray):
      target = torch.tensor(target)
  return ((pred - target.float()) ** 2).mean().item()

In [9]:
def compute_mae(pred, target):
  """Mean Absolute Error between prediction and target.

  Args:
      pred: Predictions, shape (batch, pred_len) or (num_samples, batch, pred_len).
      target: Ground truth, shape (batch, pred_len).

  Returns:
      Scalar MAE value.
  """
  pred = _to_point_forecast(pred)
  if isinstance(target, np.ndarray):
      target = torch.tensor(target)
  return (pred - target.float()).abs().mean().item()

In [10]:
def compute_metrics(pred, target):
  """Compute all metrics.

  Args:
      pred: Predictions, shape (batch, pred_len) or (num_samples, batch, pred_len).
      target: Ground truth, shape (batch, pred_len).

  Returns:
      Dict with "mse" and "mae" keys.
  """
  pred = _to_point_forecast(pred)
  if isinstance(target, np.ndarray):
      target = torch.tensor(target)
  target = target.float()

  mse = ((pred - target) ** 2).mean().item()
  mae = (pred - target).abs().mean().item()
  return {"mse": mse, "mae": mae}

## Run vanilla Chronos-2 baseline on ETTh1

In [11]:
config = {
    "ttt": {
        "n_mask": 3,
        "ttt_steps": 5,
        "lr": 1e-4
    },
    "model": {
        "name": "amazon/chronos-2",
        "device": "cuda"
    },
    "data": {
        "dataset": "ETTh1",
        "context_length": 512,
        "prediction_length": 96
    },
    "experiment": {
        "seed": 42,
        "wandb_project": "chronos-ttt"
    }
}

In [12]:
MEDIAN_QUANTILE_IDX = 10  # Chronos-2 outputs 21 quantiles [0.01, 0.05, ..., 0.5, ..., 0.99]; index 10 = 0.5
num_samples = 100 # Number of test samples to evaluate (0 = all)

model_name = config["model"]["name"]
device = config["model"]["device"]
context_length = config["data"]["context_length"]
prediction_length = config["data"]["prediction_length"]
seed = config["experiment"]["seed"]

torch.manual_seed(seed)

if device == "cuda" and not torch.cuda.is_available():
    device = "cpu"
    print("CUDA not available, falling back to CPU")

# Load model
print(f"Loading {model_name}...")
pipeline = Chronos2Pipeline.from_pretrained(
    model_name,
    device_map=device,
    torch_dtype=torch.float32,
)

# Load data
print("Loading ETTh1 dataset...")
dataset = get_etth1()
contexts, targets = prepare_data(dataset, context_length, prediction_length)

n = len(contexts) if num_samples == 0 else min(num_samples, len(contexts))
contexts = contexts[:n]
targets = targets[:n]
print(f"Evaluating on {n} samples (context_length={context_length}, "
      f"prediction_length={prediction_length})")

# Initialize wandb
wandb.init(
    project=config["experiment"]["wandb_project"],
    name="baseline",
    config={
        "method": "baseline",
        "model": model_name,
        "dataset": config["data"]["dataset"],
        "context_length": context_length,
        "prediction_length": prediction_length,
        "num_samples": n,
        "seed": seed,
    },
)

# Run predictions (batch call - pipeline handles internal batching)
print("Running predictions...")
forecasts = pipeline.predict(contexts, prediction_length=prediction_length)
# forecasts: list of n tensors, each shape (1, 21, prediction_length)

# Compute per-sample metrics
all_mse = []
all_mae = []

for i, (forecast, tgt) in enumerate(zip(forecasts, targets)):
    # forecast shape: (1, n_quantiles, pred_len) for univariate
    point_forecast = forecast[0, MEDIAN_QUANTILE_IDX, :]  # (pred_len,)
    m = compute_metrics(point_forecast.unsqueeze(0), tgt.unsqueeze(0))
    all_mse.append(m["mse"])
    all_mae.append(m["mae"])
    wandb.log({"sample_mse": m["mse"], "sample_mae": m["mae"], "sample_idx": i})

avg_mse = sum(all_mse) / len(all_mse)
avg_mae = sum(all_mae) / len(all_mae)

print(f"\n=== Baseline Results ({n} samples) ===")
print(f"MSE: {avg_mse:.6f}")
print(f"MAE: {avg_mae:.6f}")

wandb.log({"avg_mse": avg_mse, "avg_mae": avg_mae})

# Save results
results_dir = Path("results/baseline")
results_dir.mkdir(parents=True, exist_ok=True)

results = {
    "method": "baseline",
    "model": model_name,
    "dataset": config["data"]["dataset"],
    "context_length": context_length,
    "prediction_length": prediction_length,
    "num_samples": n,
    "seed": seed,
    "avg_mse": avg_mse,
    "avg_mae": avg_mae,
    "per_sample_mse": all_mse,
    "per_sample_mae": all_mae,
}

results_path = results_dir / "results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {results_path}")

wandb.finish()

Loading amazon/chronos-2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

...

wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: arman-rn (DNLP-TTSF) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Running predictions...

=== Baseline Results (100 samples) ===
MSE: 13.659108
MAE: 2.449163
Results saved to results/baseline/results.json


avg_mae,▁
avg_mse,▁
sample_idx,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇█
sample_mae,▅▃▄▁▂▂▂▁▁▃▁▂▁▂▁▁▂▁█▂▂▂▃▃▃▃▃▄▄▄▇▃▃▄▂▃▃▄▃▃
sample_mse,▂▂▂▁▃▁▁▁▂▁▁▂▁▁▆▂▁▂▂▂▂▃▁▂▃▃▅▃▇▂▄▂▁▂▂▂▂▂█▃
avg_mae,2.44916
avg_mse,13.65911
sample_idx,99
sample_mae,2.93829
sample_mse,11.84359


## Core TTT Implementation

### Masking Function

In [13]:
def create_mask(context, n_mask):
  """Create a mask tensor where last n_mask values are 0 (masked), rest are 1.

  Handles both single tensors (1D) and batched tensors (2D).

  Args:
      context: tensor of shape (context_length,) or (batch_size, context_length)
      n_mask: number of values to mask from the end

  Returns:
      mask tensor of same shape, with last n_mask values set to 0
  """
  mask = torch.ones_like(context)
  mask[..., -n_mask:] = 0
  return mask

### TTT Core Logic

In [14]:
def save_layers(layers):
  """Save state dicts for a list of (name, module) pairs.

  Returns:
      List of (module, deep-copied state_dict) tuples
  """
  return [(mod, copy.deepcopy(mod.state_dict())) for _, mod in layers]

In [15]:
def restore_layers(saved):
  """Restore modules from saved state."""
  for mod, state in saved:
      mod.load_state_dict(state)

In [16]:
def get_ttt_layers(model, target):
  """Return list of (name, module) pairs for the TTT target layer(s).

  Args:
      model: Chronos2Model instance (pipeline.model)
      target: which layer(s) to adapt:
          "input"  — input_patch_embedding only
          "output" — output_patch_embedding only
          "both"   — both input and output patch embeddings

  Returns:
      List of (name, module) tuples
  """
  if target == "input":
      return [("input_patch_embedding", model.input_patch_embedding)]
  elif target == "output":
      return [("output_patch_embedding", model.output_patch_embedding)]
  elif target == "both":
      return [
          ("input_patch_embedding", model.input_patch_embedding),
          ("output_patch_embedding", model.output_patch_embedding),
      ]
  else:
      raise ValueError(f"Unknown target '{target}', expected 'input', 'output', or 'both'")

In [17]:
def ttt_step(model, context, n_mask, optimizer):
  """One TTT optimization step.

  Splits context into practice (input) and target (last n_mask values).
  Uses the model's built-in forward with future_target, which computes
  quantile loss in the normalized space — matching how the model was
  pretrained.

  Args:
      model: Chronos2Model instance (pipeline.model)
      context: (batch_size, context_length) tensor
      n_mask: number of values to mask from the end
      optimizer: optimizer for target layer parameters

  Returns:
      loss value (float)
  """
  practice_context = context[:, :-n_mask]
  target = context[:, -n_mask:]

  output_patch_size = model.chronos_config.output_patch_size
  num_output_patches = math.ceil(n_mask / output_patch_size)

  optimizer.zero_grad()

  output = model(
      context=practice_context,
      future_target=target,
      num_output_patches=num_output_patches,
  )

  loss = output.loss
  loss.backward()
  optimizer.step()

  return loss.item()

In [18]:
class TTTChronos:
  """Chronos-2 wrapper with Test-Time Training — configurable target layer.

  Before making a prediction, adapts the chosen layer(s) by running a
  self-supervised "practice exam" on the recent context: mask the last
  n_mask known values, predict them, and update weights to minimize
  prediction error. Then predict the actual future with adapted weights,
  and reset afterwards.

  Usage:
      model = TTTChronos(pipeline, n_mask=16, ttt_steps=5, lr=1e-4, target="output")
      forecast = model.predict(context, prediction_length=96)
  """

  def __init__(self, pipeline, n_mask=16, ttt_steps=5, lr=1e-4, target="output",
                optimizer="adam"):
      self.pipeline = pipeline
      self.n_mask = n_mask
      self.ttt_steps = ttt_steps
      self.lr = lr
      self.target = target
      self.optimizer_type = optimizer

  def predict(self, context, prediction_length):
      """Predict with TTT adaptation on the chosen target layer(s).

      Args:
          context: input time series, shape (context_length,) or
                    (batch_size, context_length)
          prediction_length: number of future steps to forecast

      Returns:
          forecast from pipeline.predict() - list of tensors,
          each of shape (n_variates, n_quantiles, prediction_length)
      """
      model = self.pipeline.model
      original_context = context

      if context.ndim == 1:
          context = context.unsqueeze(0)
      context = context.to(device=model.device, dtype=torch.float32)

      # 1. Get target layers and save their state
      layers = get_ttt_layers(model, self.target)
      saved = save_layers(layers)

      # 2. Freeze all params, enable only target layer(s)
      for param in model.parameters():
          param.requires_grad_(False)
      for _, mod in layers:
          mod.requires_grad_(True)

      trainable_params = []
      for _, mod in layers:
          trainable_params.extend(mod.parameters())

      if self.optimizer_type == "sgd":
          optimizer = torch.optim.SGD(trainable_params, lr=self.lr)
      else:
          optimizer = torch.optim.Adam(trainable_params, lr=self.lr)

      # 3. TTT adaptation loop
      model.train()
      for step in range(self.ttt_steps):
          loss = ttt_step(model, context, self.n_mask, optimizer)
          print(f"  TTT step {step}: loss={loss:.4f}")

      # 4. Inference with adapted weights
      model.eval()

      if original_context.ndim == 1:
          pipeline_input = [original_context]
      elif original_context.ndim == 2:
          pipeline_input = [original_context[i] for i in range(original_context.shape[0])]
      else:
          pipeline_input = original_context

      forecast = self.pipeline.predict(pipeline_input, prediction_length=prediction_length)

      # 5. Reset weights and requires_grad state
      restore_layers(saved)
      for param in model.parameters():
          param.requires_grad_(True)

      return forecast

### Smoke test for TTT implementation on Chronos-2

In [19]:
def make_synthetic_context(length=512, seed=42):
  """Sine wave + linear trend + noise. More meaningful than pure random."""
  rng = np.random.default_rng(seed)
  t = np.arange(length, dtype=np.float32)
  series = (
      10.0 * np.sin(2 * np.pi * t / 50)
      + 0.01 * t
      + rng.normal(0, 0.5, length).astype(np.float32)
  )
  return torch.tensor(series)

In [20]:
def load_etth1_context(context_length):
  """Load first context_length values from ETTh1 test set."""

  dataset = get_etth1()
  entry = list(dataset.test)[0]
  ts = np.array(entry["target"], dtype=np.float32)
  return torch.tensor(ts[:context_length])

In [21]:
use_etth1 = True

n_mask = config["ttt"]["n_mask"]
ttt_steps = config["ttt"]["ttt_steps"]
lr = config["ttt"]["lr"]
model_name = config["model"]["name"]
device = config["model"]["device"]
context_length = config["data"]["context_length"]
prediction_length = config["data"]["prediction_length"]
seed = config["experiment"]["seed"]

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Load model ----
print(f"Loading Chronos-2 from amazon/chronos-2 on {device} ...")
pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=device,
    torch_dtype=torch.float32,
)
model = pipeline.model
print(f"  Model class : {type(model).__name__}")
print(f"  Patch sizes : input={model.chronos_config.input_patch_size}, "
      f"output={model.chronos_config.output_patch_size}")
print(f"  Quantiles   : {model.chronos_config.quantiles}")
total_params = sum(p.numel() for p in model.parameters())
out_emb_params = sum(p.numel() for p in model.output_patch_embedding.parameters())
in_emb_params = sum(p.numel() for p in model.input_patch_embedding.parameters())
print(f"  Total params: {total_params:,}")
print(f"  Input embedding params:  {in_emb_params:,} ({in_emb_params / total_params * 100:.1f}%)")
print(f"  Output embedding params: {out_emb_params:,} ({out_emb_params / total_params * 100:.1f}%)")

# ---- Prepare context ----
if use_etth1:
    print(f"\nLoading ETTh1 context (first {context_length} values) ...")
    context = load_etth1_context(context_length)
else:
    print(f"\nUsing synthetic context (sine + trend + noise, length={context_length}) ...")
    context = make_synthetic_context(length=context_length)

print(f"  shape={tuple(context.shape)}, dtype={context.dtype}")
print(f"  range=[{context.min():.4f}, {context.max():.4f}], "
      f"mean={context.mean():.4f}, std={context.std():.4f}")

# ---- Snapshot output_patch_embedding BEFORE TTT ----
pre_state = copy.deepcopy(model.output_patch_embedding.state_dict())

# ---- Run TTT ----
print(f"\n{'='*60}")
print(f"Running TTT  n_mask={n_mask}  steps={ttt_steps}  lr={lr}")
print(f"{'='*60}")

ttt_model = TTTChronos(
    pipeline,
    n_mask=n_mask,
    ttt_steps=ttt_steps,
    lr=lr,
)
forecast = ttt_model.predict(context, prediction_length=prediction_length)

# ---- Inspect forecast ----
print(f"\n{'='*60}")
print("Forecast results")
print(f"{'='*60}")
print(f"  pipeline.predict() returned {len(forecast)} tensor(s)")
for i, f in enumerate(forecast):
    print(f"  forecast[{i}]: shape={tuple(f.shape)}, dtype={f.dtype}")
    # Expected shape: (n_variates, n_quantiles, prediction_length)
    if f.ndim == 3:
        quantiles = model.chronos_config.quantiles
        if 0.5 in quantiles:
            median_idx = quantiles.index(0.5)
        else:
            median_idx = len(quantiles) // 2
        median = f[0, median_idx]  # first variate, median quantile
        print(f"    median (q=0.5) range: [{median.min():.4f}, {median.max():.4f}]")

# ---- Verify embedding reset ----
print(f"\n{'='*60}")
print("Embedding reset verification")
print(f"{'='*60}")
post_state = model.output_patch_embedding.state_dict()

all_match = True
for key in pre_state:
    if not torch.equal(pre_state[key], post_state[key]):
        all_match = False
        diff = (pre_state[key] - post_state[key]).abs().max().item()
        print(f"  MISMATCH  '{key}': max abs diff = {diff:.6e}")

if all_match:
    print("  PASS  All embedding weights restored to pre-TTT values.")
else:
    print("  FAIL  Embedding weights differ from pre-TTT snapshot!")

print("\nDone.")

Loading Chronos-2 from amazon/chronos-2 on cuda ...
  Model class : Chronos2Model
  Patch sizes : input=16, output=16
  Quantiles   : [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]
  Total params: 119,477,664
  Input embedding params:  2,548,224 (2.1%)
  Output embedding params: 3,653,280 (3.1%)

Loading ETTh1 context (first 512 values) ...
  shape=(512,), dtype=torch.float32
  range=[4.2200, 19.1560], mean=10.4040, std=2.5943

Running TTT  n_mask=3  steps=5  lr=0.0001
  TTT step 0: loss=0.4671
  TTT step 1: loss=0.4349
  TTT step 2: loss=0.3776
  TTT step 3: loss=0.3228
  TTT step 4: loss=0.2580

Forecast results
  pipeline.predict() returned 1 tensor(s)
  forecast[0]: shape=(1, 21, 96), dtype=torch.float32
    median (q=0.5) range: [7.7147, 14.5230]

Embedding reset verification
  PASS  All embedding weights restored to pre-TTT values.

Done.


## Run TTT (Test-Time Training) experiment on ETTh1

In [23]:
MEDIAN_QUANTILE_IDX = 10  # Chronos-2 outputs 21 quantiles [0.01, 0.05, ..., 0.5, ..., 0.99]; index 10 = 0.5
num_samples = 100 # Number of test samples to evaluate (0 = all)

n_mask = config["ttt"]["n_mask"]
ttt_steps = config["ttt"]["ttt_steps"]
lr = config["ttt"]["lr"]
target = "output" # ["input", "output", "both"]
optim = "adam" #["adam", "sgd"]
model_name = config["model"]["name"]
device = config["model"]["device"]
context_length = config["data"]["context_length"]
prediction_length = config["data"]["prediction_length"]
seed = config["experiment"]["seed"]

torch.manual_seed(seed)

if device == "cuda" and not torch.cuda.is_available():
    device = "cpu"
    print("CUDA not available, falling back to CPU")

# ---- Load model ----
print(f"Loading {model_name}...")
pipeline = Chronos2Pipeline.from_pretrained(
    model_name,
    device_map=device,
    torch_dtype=torch.float32,
)

ttt_model = TTTChronos(
    pipeline, n_mask=n_mask, ttt_steps=ttt_steps, lr=lr, target=target,
    optimizer=optim
)

# ---- Load data ----
print("Loading ETTh1 dataset...")
dataset = get_etth1()
contexts, targets = prepare_data(dataset, context_length, prediction_length)

n = len(contexts) if num_samples == 0 else min(num_samples, len(contexts))
contexts = contexts[:n]
targets = targets[:n]
print(f"Evaluating on {n} samples (context_length={context_length}, "
      f"prediction_length={prediction_length})")
print(f"TTT config: target={target}, n_mask={n_mask}, ttt_steps={ttt_steps}, lr={lr}, optimizer={optim}")

# ---- Initialize wandb ----
wandb.init(
    project=config["experiment"]["wandb_project"],
    name=f"ttt_{target}_{optim}_mask{n_mask}_steps{ttt_steps}",
    config={
        "method": "ttt",
        "target": target,
        "optimizer": optim,
        "model": model_name,
        "dataset": config["data"]["dataset"],
        "context_length": context_length,
        "prediction_length": prediction_length,
        "n_mask": n_mask,
        "ttt_steps": ttt_steps,
        "lr": lr,
        "num_samples": n,
        "seed": seed,
    },
)

# ---- Run TTT predictions per sample ----
all_mse = []
all_mae = []

pbar = tqdm(
    enumerate(zip(contexts, targets)),
    total=n,
    desc=f"TTT ({target}, mask={n_mask}, steps={ttt_steps})",
)
for i, (ctx, tgt) in pbar:
    with contextlib.redirect_stdout(io.StringIO()):
        forecast = ttt_model.predict(ctx, prediction_length=prediction_length)

    point_forecast = forecast[0][0, MEDIAN_QUANTILE_IDX, :]
    m = compute_metrics(point_forecast.unsqueeze(0), tgt.unsqueeze(0))
    all_mse.append(m["mse"])
    all_mae.append(m["mae"])

    wandb.log({"sample_mse": m["mse"], "sample_mae": m["mae"], "sample_idx": i})

    running_mse = sum(all_mse) / len(all_mse)
    running_mae = sum(all_mae) / len(all_mae)
    pbar.set_postfix(mse=f"{running_mse:.4f}", mae=f"{running_mae:.4f}")

# ---- Final metrics ----
avg_mse = sum(all_mse) / len(all_mse)
avg_mae = sum(all_mae) / len(all_mae)

print(f"\n=== TTT Results ({n} samples) ===")
print(f"  target={target}, n_mask={n_mask}, ttt_steps={ttt_steps}, lr={lr}")
print(f"  MSE: {avg_mse:.6f}")
print(f"  MAE: {avg_mae:.6f}")

wandb.log({"avg_mse": avg_mse, "avg_mae": avg_mae})

# ---- Compare with baseline ----
baseline_path = Path("results/baseline/results.json")
if baseline_path.exists():
    with open(baseline_path) as f:
        baseline = json.load(f)

    b_mse = baseline["avg_mse"]
    b_mae = baseline["avg_mae"]

    mse_diff = avg_mse - b_mse
    mae_diff = avg_mae - b_mae
    mse_pct = mse_diff / b_mse * 100
    mae_pct = mae_diff / b_mae * 100

    print(f"\n=== Comparison with Baseline ===")
    print(f"  {'Metric':<6} {'Baseline':>12} {'TTT':>12} {'Diff':>12} {'Change':>8}")
    print(f"  {'MSE':<6} {b_mse:>12.6f} {avg_mse:>12.6f} {mse_diff:>+12.6f} {mse_pct:>+7.1f}%")
    print(f"  {'MAE':<6} {b_mae:>12.6f} {avg_mae:>12.6f} {mae_diff:>+12.6f} {mae_pct:>+7.1f}%")

    wandb.log({
        "baseline_mse": b_mse,
        "baseline_mae": b_mae,
        "mse_change_pct": mse_pct,
        "mae_change_pct": mae_pct,
    })
else:
    print(f"\n(No baseline results found at {baseline_path}. "
          "Run experiments/run_baseline.py first for comparison.)")

# ---- Save results ----
results_dir = Path("results/ttt")
results_dir.mkdir(parents=True, exist_ok=True)

results = {
    "method": "ttt",
    "target": target,
    "optimizer": optim,
    "model": model_name,
    "dataset": config["data"]["dataset"],
    "context_length": context_length,
    "prediction_length": prediction_length,
    "n_mask": n_mask,
    "ttt_steps": ttt_steps,
    "lr": lr,
    "num_samples": n,
    "seed": seed,
    "avg_mse": avg_mse,
    "avg_mae": avg_mae,
    "per_sample_mse": all_mse,
    "per_sample_mae": all_mae,
}

results_path = results_dir / f"results_{target}_{optim}_{n_mask}_{ttt_steps}.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {results_path}")

wandb.finish()

Loading amazon/chronos-2...
Loading ETTh1 dataset...
Evaluating on 100 samples (context_length=512, prediction_length=96)
TTT config: target=output, n_mask=3, ttt_steps=5, lr=0.0001, optimizer=adam


TTT (output, mask=3, steps=5): 100%|██████████| 100/100 [00:49<00:00,  2.01it/s, mae=2.4667, mse=13.8142]


=== TTT Results (100 samples) ===
  target=output, n_mask=3, ttt_steps=5, lr=0.0001
  MSE: 13.814153
  MAE: 2.466696

=== Comparison with Baseline ===
  Metric     Baseline          TTT         Diff   Change
  MSE       13.659108    13.814153    +0.155045    +1.1%
  MAE        2.449163     2.466696    +0.017533    +0.7%

Results saved to results/ttt/results_output_adam_3_5.json


avg_mae,▁
avg_mse,▁
baseline_mae,▁
baseline_mse,▁
mae_change_pct,▁
mse_change_pct,▁
sample_idx,▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇███
sample_mae,▄▃▃▂▁▁▃▁▂▁▁▁▁▂▂▁▃▃▂▃▃▂▃▄▄▅▇▃▂▄▃▂▃▄▃▃▄█▃▃
sample_mse,▃▂▂▁▃▂▁▁▁▂▁▁▁▁█▂▁▂▂▁▁▂▂▄▂▂▄▆▄▂▄▅▃▃▄▂▃▂▃▄
avg_mae,2.4667
avg_mse,13.81415


## Run TTT hyperparameter ablation sweep on ETTh1


In [27]:
MEDIAN_QUANTILE_IDX = 10  # 21 quantiles; index 10 = 0.5 median

# ---- Hyperparameter grid ----
GRID = {
  "target": ["output"],
  "n_mask": [16, 32],
  "ttt_steps": [1, 2, 5],
  "lr": [1e-5, 5e-5, 1e-4, 5e-4],
  "optimizer": ["adam", "sgd"],
}

In [28]:
def grid_configs(grid):
  """Expand grid dict into list of config dicts."""
  keys = list(grid.keys())
  values = list(grid.values())
  configs = []
  for combo in product(*values):
      configs.append(dict(zip(keys, combo)))
  return configs

In [29]:
def run_single_config(pipeline, contexts, targets, prediction_length, config):
  """Run TTT with a single hyperparameter config. Returns avg metrics."""
  ttt_model = TTTChronos(
      pipeline,
      n_mask=config["n_mask"],
      ttt_steps=config["ttt_steps"],
      lr=config["lr"],
      target=config["target"],
      optimizer=config["optimizer"],
  )

  all_mse = []
  all_mae = []

  for ctx, tgt in zip(contexts, targets):
      with contextlib.redirect_stdout(io.StringIO()):
          forecast = ttt_model.predict(ctx, prediction_length=prediction_length)

      point_forecast = forecast[0][0, MEDIAN_QUANTILE_IDX, :]
      m = compute_metrics(point_forecast.unsqueeze(0), tgt.unsqueeze(0))
      all_mse.append(m["mse"])
      all_mae.append(m["mae"])

  avg_mse = sum(all_mse) / len(all_mse)
  avg_mae = sum(all_mae) / len(all_mae)

  return {
      "avg_mse": avg_mse,
      "avg_mae": avg_mae,
      "per_sample_mse": all_mse,
      "per_sample_mae": all_mae,
  }

In [30]:
num_samples = 100 # Number of test samples to evaluate (0 = all)
model_name = config["model"]["name"]
device = config["model"]["device"]
context_length = config["data"]["context_length"]
prediction_length = config["data"]["prediction_length"]
seed = config["experiment"]["seed"]

torch.manual_seed(seed)

if device == "cuda" and not torch.cuda.is_available():
    device = "cpu"
    print("CUDA not available, falling back to CPU")

# ---- Load model ONCE ----
print(f"Loading {model_name}...")
pipeline = Chronos2Pipeline.from_pretrained(
    model_name,
    device_map=device,
    torch_dtype=torch.float32,
)

# ---- Load data ONCE ----
print("Loading ETTh1 dataset...")
dataset = get_etth1()
contexts, targets = prepare_data(dataset, context_length, prediction_length)

n = len(contexts) if num_samples == 0 else min(num_samples, len(contexts))
contexts = contexts[:n]
targets = targets[:n]

# ---- Generate all configs ----
all_configs = grid_configs(GRID)
total = len(all_configs)
print(f"\nAblation sweep: {total} configurations x {n} samples")
print(f"Grid: {GRID}\n")

# ---- Load baseline for comparison ----
baseline_mse = None
baseline_mae = None
baseline_path = Path("results/baseline/results.json")
if baseline_path.exists():
    with open(baseline_path) as f:
        baseline = json.load(f)
    baseline_mse = baseline["avg_mse"]
    baseline_mae = baseline["avg_mae"]
    print(f"Baseline: MSE={baseline_mse:.6f}, MAE={baseline_mae:.6f}\n")

# ---- Initialize wandb for the sweep ----
wandb.init(
    project=config["experiment"]["wandb_project"],
    name="ablation_sweep",
    config={
        "method": "ablation",
        "grid": GRID,
        "model": model_name,
        "dataset": config["data"]["dataset"],
        "context_length": context_length,
        "prediction_length": prediction_length,
        "num_samples": n,
        "seed": seed,
    },
)

# ---- Run sweep ----
all_results = []

pbar = tqdm(all_configs, desc="Ablation sweep", total=total)
for i, cfg in enumerate(pbar):
    label = (f"target={cfg['target']}, n_mask={cfg['n_mask']}, "
              f"steps={cfg['ttt_steps']}, lr={cfg['lr']}, opt={cfg['optimizer']}")
    pbar.set_description(f"[{i+1}/{total}] {label}")

    metrics = run_single_config(
        pipeline, contexts, targets, prediction_length, cfg
    )

    result = {**cfg, **metrics}

    # Add baseline comparison
    if baseline_mse is not None:
        result["mse_change_pct"] = (metrics["avg_mse"] - baseline_mse) / baseline_mse * 100
        result["mae_change_pct"] = (metrics["avg_mae"] - baseline_mae) / baseline_mae * 100

    all_results.append(result)

    # Log to wandb
    wandb.log({
        "config_idx": i,
        "target": cfg["target"],
        "n_mask": cfg["n_mask"],
        "ttt_steps": cfg["ttt_steps"],
        "lr": cfg["lr"],
        "optimizer": cfg["optimizer"],
        "avg_mse": metrics["avg_mse"],
        "avg_mae": metrics["avg_mae"],
        **({"mse_change_pct": result["mse_change_pct"],
            "mae_change_pct": result["mae_change_pct"]}
            if baseline_mse is not None else {}),
    })

    # Update progress bar with running best
    best_so_far = min(all_results, key=lambda r: r["avg_mse"])
    pbar.set_postfix(
        current_mse=f"{metrics['avg_mse']:.4f}",
        best_mse=f"{best_so_far['avg_mse']:.4f}",
    )

# ---- Summary table ----
# Sort by MSE ascending
all_results.sort(key=lambda r: r["avg_mse"])

print(f"\n{'='*100}")
print(f"ABLATION RESULTS ({total} configs, {n} samples)")
print(f"{'='*100}")

header = (f"  {'Rank':<5} {'Target':<8} {'n_mask':<7} {'Steps':<6} "
          f"{'LR':<10} {'Opt':<6} {'MSE':>10} {'MAE':>10}")
if baseline_mse is not None:
    header += f" {'MSE %':>8}"
print(header)
print(f"  {'-'*len(header.strip())}")

if baseline_mse is not None:
    print(f"  {'base':<5} {'--':<8} {'--':<7} {'--':<6} "
          f"{'--':<10} {'--':<6} {baseline_mse:>10.4f} {baseline_mae:>10.4f} {'0.0%':>8}")

for rank, r in enumerate(all_results, 1):
    line = (f"  {rank:<5} {r['target']:<8} {r['n_mask']:<7} {r['ttt_steps']:<6} "
            f"{r['lr']:<10.0e} {r['optimizer']:<6} {r['avg_mse']:>10.4f} {r['avg_mae']:>10.4f}")
    if baseline_mse is not None:
        line += f" {r['mse_change_pct']:>+7.1f}%"
    print(line)

# ---- Best config ----
best = all_results[0]
print(f"\n{'='*100}")
print(f"BEST CONFIG:")
print(f"  target={best['target']}, n_mask={best['n_mask']}, "
      f"ttt_steps={best['ttt_steps']}, lr={best['lr']}, optimizer={best['optimizer']}")
print(f"  MSE: {best['avg_mse']:.6f}  MAE: {best['avg_mae']:.6f}")
if baseline_mse is not None:
    print(f"  vs Baseline: MSE {best['mse_change_pct']:+.1f}%, MAE {best['mae_change_pct']:+.1f}%")
print(f"{'='*100}")

wandb.log({
    "best_mse": best["avg_mse"],
    "best_mae": best["avg_mae"],
    "best_target": best["target"],
    "best_n_mask": best["n_mask"],
    "best_ttt_steps": best["ttt_steps"],
    "best_lr": best["lr"],
    "best_optimizer": best["optimizer"],
})

# ---- Save results ----
results_dir = Path("results/ablations")
results_dir.mkdir(parents=True, exist_ok=True)

# Strip per-sample lists for the summary file (keeps it small)
summary_results = []
for r in all_results:
    summary = {k: v for k, v in r.items()
                if k not in ("per_sample_mse", "per_sample_mae")}
    summary_results.append(summary)

output = {
    "grid": GRID,
    "num_samples": n,
    "baseline_mse": baseline_mse,
    "baseline_mae": baseline_mae,
    "best": summary_results[0],
    "all_results": summary_results,
}

results_path = results_dir / "all_results.json"
with open(results_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"\nResults saved to {results_path}")

wandb.finish()

Loading amazon/chronos-2...
Loading ETTh1 dataset...

Ablation sweep: 48 configurations x 100 samples
Grid: {'target': ['output'], 'n_mask': [16, 32], 'ttt_steps': [1, 2, 5], 'lr': [1e-05, 5e-05, 0.0001, 0.0005], 'optimizer': ['adam', 'sgd']}

Baseline: MSE=13.659108, MAE=2.449163



[48/48] target=output, n_mask=32, steps=5, lr=0.0005, opt=sgd: 100%|██████████| 48/48 [10:07<00:00, 12.66s/it, best_mse=13.6589, current_mse=13.9869]


ABLATION RESULTS (48 configs, 100 samples)
  Rank  Target   n_mask  Steps  LR         Opt           MSE        MAE    MSE %
  ------------------------------------------------------------------------------
  base  --       --      --     --         --        13.6591     2.4492     0.0%
  1     output   16      1      1e-04      sgd       13.6589     2.4490    -0.0%
  2     output   16      2      1e-05      sgd       13.6592     2.4491    +0.0%
  3     output   16      5      1e-05      sgd       13.6592     2.4491    +0.0%
  4     output   16      1      1e-05      sgd       13.6592     2.4491    +0.0%
  5     output   32      1      1e-05      sgd       13.6597     2.4491    +0.0%
  6     output   16      1      5e-05      sgd       13.6599     2.4491    +0.0%
  7     output   16      2      5e-05      sgd       13.6600     2.4490    +0.0%
  8     output   16      1      1e-05      adam      13.6603     2.4490    +0.0%
  9     output   32      2      1e-05      sgd       13.6604     

avg_mae,▁▁▁▁▁▃▁▁▁▁▁▁▄▁▁▁▁▂▁▅▁▁▁▁▁▄▁▁▁▁▁▁█▁▁▂▁▃▁▁
avg_mse,▁▁▁▁▁▂▁▁▁▁▁▁▃▁▁▁▁▂▁▄▁▁▁▁▁▄▁▁▁▁▂▁█▁▁▂▁▃▁▁
best_lr,▁
best_mae,▁
best_mse,▁
best_n_mask,▁
best_ttt_steps,▁
config_idx,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
lr,▁▁▂▂▂██▁▁▂▂▂██▁▂▂▂▂█▁▁▂▂▂██▁▁▂▂▂██▁▂▂▂▂█
mae_change_pct,▁▁▁▁▁▃▁▁▁▁▁▁▄▁▁▁▁▂▁▅▁▁▁▁▁▄▁▁▁▁▁▁█▁▁▂▁▃▁▁
+3,...
